#Downloads and imports (~30s)

In [ ]:
%load_ext cuml.accel

In [ ]:
# download TA-Lib
!wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
#!ls
!tar xvzf ta-lib-0.4.0-src.tar.gz
#!ls
import os
os.chdir('ta-lib') # Can't use !cd in co-lab
!./configure --prefix=/usr
!make
!make install
# wait ~ 30s
os.chdir('../')
#!ls
!pip install -q TA-Lib

ta-lib/
ta-lib/config.sub
ta-lib/aclocal.m4
ta-lib/CHANGELOG.TXT
ta-lib/include/
ta-lib/include/ta_abstract.h
ta-lib/include/ta_func.h
ta-lib/include/ta_common.h
ta-lib/include/ta_config.h.in
ta-lib/include/Makefile.am
ta-lib/include/ta_libc.h
ta-lib/include/ta_defs.h
ta-lib/missing
ta-lib/ta-lib.spec.in
ta-lib/config.guess
ta-lib/Makefile.in
ta-lib/ta-lib.dpkg.in
ta-lib/Makefile.am
ta-lib/autogen.sh
ta-lib/install-sh
ta-lib/configure
ta-lib/depcomp
ta-lib/HISTORY.TXT
ta-lib/configure.in
ta-lib/autom4te.cache/
ta-lib/autom4te.cache/output.0
ta-lib/autom4te.cache/requests
ta-lib/autom4te.cache/output.1
ta-lib/autom4te.cache/traces.0
ta-lib/autom4te.cache/traces.1
ta-lib/ltmain.sh
ta-lib/ta-lib-config.in
ta-lib/src/
ta-lib/src/ta_func/
ta-lib/src/ta_func/ta_MACDFIX.c
ta-lib/src/ta_func/ta_CDLPIERCING.c
ta-lib/src/ta_func/ta_DIV.c
ta-lib/src/ta_func/ta_ROCR100.c
ta-lib/src/ta_func/ta_ADXR.c
ta-lib/src/ta_func/ta_MAVP.c
ta-lib/src/ta_func/ta_CDLCLOSINGMARUBOZU.c
ta-lib/src/ta_func/ta_COSH.

In [ ]:
!pip install scikit-tree

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 14.5 MB/s eta 0:00:00


In [ ]:
import talib as ta
import random
import time
import datetime as dt
from datetime import datetime, timedelta
import numpy as np
import yfinance as yf
import pandas as pd
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning) #ignores "Dataframe is highly fragmented" warning.
from sklearn import preprocessing
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score,mean_absolute_error, mean_absolute_percentage_error, r2_score, precision_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sktree import tree
from sklearn.ensemble import (AdaBoostClassifier, RandomForestClassifier,
                              StackingClassifier, GradientBoostingClassifier,
                              ExtraTreesClassifier, BaggingClassifier, VotingClassifier, IsolationForest)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier, MLPRegressor
# Import regressors
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFECV, SelectFromModel
from matplotlib import pyplot as plt
import math
import lightgbm as lgb
from dateutil.relativedelta import relativedelta

#Functions (~10s)


In [ ]:
def change_days(date_str, days):
    import datetime as dt
    date = dt.datetime.strptime(date_str, "%Y-%m-%d")

    new_date = date + timedelta(days=days)

    return new_date.strftime("%Y-%m-%d")

def change_months(date_str, monthss):
  import datetime as dt
  date_obj = datetime.strptime(date_str, "%Y-%m-%d")

  new_date = date_obj + relativedelta(months=monthss)

  return new_date.strftime("%Y-%m-%d")

def data(stock, startdate, enddate, shuffle=True):

  #push end date back a month to increase viable data(non-NaN) to right before dates tested
  enddate = enddate

  stock_data = yf.Ticker(stock).history(start = startdate, end=enddate)

  #Closing prices on previous days:
  for i in range(1, 60):
          stock_data[f"Close -{i}d"] = stock_data["Close"].shift(i)
          stock_data[f"High -{i}d"] = stock_data["High"].shift(i)
          stock_data[f"Low -{i}d"] = stock_data["Low"].shift(i)


  #Technical indicators
  stock_data['BB_upper'], stock_data['BB_middle'], stock_data['BB_lower'] = ta.BBANDS(stock_data['Close'], timeperiod=20, nbdevup=0, nbdevdn=2, matype=0)
  stock_data['KAMA'] = ta.KAMA(stock_data['Close'], timeperiod=30)
  stock_data['MA'] = ta.MA(stock_data['Close'], timeperiod=30)
  stock_data['TRIMA'] = ta.TRIMA(stock_data['Close'], timeperiod=30)
  stock_data['ADXR'] = ta.ADXR(stock_data['High'], stock_data['Low'], stock_data['Close'], timeperiod=14)
  stock_data['APO'] = ta.APO(stock_data['Close'], fastperiod=12, slowperiod=26)
  stock_data['MINUS_DI'] = ta.MINUS_DI(stock_data['High'], stock_data['Low'], stock_data['Close'], timeperiod=14)
  stock_data['MINUS_DM'] = ta.MINUS_DM(stock_data['High'], stock_data['Low'], timeperiod=14)
  stock_data['PPO'] = ta.PPO(stock_data['Close'], fastperiod=12, slowperiod=26, matype=0)
  stock_data['ADOSC'] = ta.ADOSC(stock_data['High'], stock_data['Low'], stock_data['Close'], stock_data['Volume'], fastperiod=3, slowperiod=10)
  stock_data['HT_DCPERIOD'] = ta.HT_DCPERIOD(stock_data['Close'])
  stock_data['TYPPRICE'] = ta.TYPPRICE(stock_data['High'], stock_data['Low'], stock_data['Close'])
  stock_data['WCLPRICE'] = ta.WCLPRICE(stock_data['High'], stock_data['Low'], stock_data['Close'])
  stock_data['DEMA'] = ta.DEMA(stock_data['Close'], timeperiod=30)##
  stock_data["EMA"] = ta.EMA(stock_data["Close"], timeperiod=14)  ##
  stock_data['HT_TRENDLINE'] = ta.HT_TRENDLINE(stock_data['Close'])##
  stock_data['MIDPOINT'] = ta.MIDPOINT(stock_data['Close'], timeperiod=14)##
  stock_data['ADX'] = ta.ADX(stock_data["High"], stock_data["Low"], stock_data["Close"], timeperiod=14)  ##
  stock_data["RSI"] = ta.RSI(stock_data["Close"], timeperiod=14)##
  stock_data["OBV"] = ta.OBV(stock_data["Close"], stock_data["Volume"])##
  stock_data["MIDPOINT_30"] = stock_data["MIDPOINT"].pct_change(periods=30)
  stock_data['SAR'] = ta.SAR(stock_data['High'], stock_data['Low'], acceleration=0.02, maximum=0.2)
  stock_data['macd'], stock_data['macd_signal'], stock_data['macd_diff'] = ta.MACD(stock_data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)
  stock_data["bband_width"] = stock_data['BB_upper'] - stock_data['BB_lower']
  stock_data['stochastic oscillator %K'], stock_data['stochastic oscillator %D'] = ta.STOCH(stock_data['High'], stock_data['Low'], stock_data['Close'], fastk_period=5, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)
  stock_data["atr"] = ta.ATR(stock_data["High"], stock_data["Low"], stock_data["Close"], timeperiod=14)
  stock_data["SMA"] = ta.SMA(stock_data["Close"], timeperiod=14)
  stock_data["ROC"] = ta.ROC(stock_data["Close"], timeperiod=12)
  stock_data["CCI"] = ta.CCI(stock_data["High"], stock_data["Low"], stock_data["Close"], timeperiod=14)
  stock_data["AD"] = ta.AD(stock_data["High"], stock_data["Low"], stock_data["Close"], stock_data["Volume"])
  stock_data["NATR"] = ta.NATR(stock_data["High"], stock_data["Low"], stock_data["Close"], timeperiod=14)
  stock_data["TRIX"] = ta.TRIX(stock_data["Close"],timeperiod=30)

  #5 day
  stock_data['BB_upper_5'], stock_data['BB_middle_5'], stock_data['BB_lower_5'] = ta.BBANDS(stock_data['Close -5d'], timeperiod=5, nbdevup=0, nbdevdn=2, matype=0)
  stock_data["bband_width_5"] = stock_data['BB_upper_5'] - stock_data['BB_lower_5']
  stock_data['KAMA_5'] = ta.KAMA(stock_data['Close -5d'], timeperiod=5)
  stock_data['MA_5'] = ta.MA(stock_data['Close -5d'], timeperiod=5)
  stock_data['TRIMA_5'] = ta.TRIMA(stock_data['Close -5d'], timeperiod=5)
  stock_data['ADXR_5'] = ta.ADXR(stock_data['High -5d'], stock_data['Low -5d'], stock_data['Close -5d'], timeperiod=5)
  stock_data['MINUS_DI_5'] = ta.MINUS_DI(stock_data['High -5d'], stock_data['Low -5d'], stock_data['Close -5d'], timeperiod=5)
  stock_data['MINUS_DM_5'] = ta.MINUS_DM(stock_data['High -5d'], stock_data['Low -5d'], timeperiod=5)
  stock_data['DEMA_5'] = ta.DEMA(stock_data['Close -5d'], timeperiod=5)
  stock_data["EMA_5"] = ta.EMA(stock_data["Close -5d"], timeperiod=5)
  stock_data['HT_TRENDLINE_5'] = stock_data["HT_TRENDLINE"].pct_change(periods=5)
  stock_data['MIDPOINT_5'] = ta.MIDPOINT(stock_data['Close -5d'], timeperiod=5)
  stock_data['ADX_5'] = ta.ADX(stock_data["High -5d"], stock_data["Low -5d"], stock_data["Close -5d"], timeperiod=5)
  stock_data["RSI_5"] = ta.RSI(stock_data["Close -5d"], timeperiod=5)
  stock_data["MIDPOINT_30_5"] = stock_data["MIDPOINT_5"].pct_change(periods=5)
  stock_data["atr_5"] = ta.ATR(stock_data["High -5d"], stock_data["Low -5d"], stock_data["Close -5d"], timeperiod=5)
  stock_data["SMA_5"] = ta.SMA(stock_data["Close -5d"], timeperiod=5)
  stock_data["ROC_5"] = ta.ROC(stock_data["Close -5d"], timeperiod=5)
  stock_data["CCI_5"] = ta.CCI(stock_data["High -5d"], stock_data["Low -5d"], stock_data["Close -5d"], timeperiod=5)
  stock_data["NATR_5"] = ta.NATR(stock_data["High -5d"], stock_data["Low -5d"], stock_data["Close -5d"], timeperiod=5)
  stock_data["TRIX_5"] = ta.TRIX(stock_data["Close -5d"], timeperiod=5)

  #10 day
  stock_data['BB_upper_10'], stock_data['BB_middle_10'], stock_data['BB_lower_10'] = ta.BBANDS(stock_data['Close -10d'], timeperiod=5, nbdevup=0, nbdevdn=2, matype=0)
  stock_data["bband_width_10"] = stock_data['BB_upper_10'] - stock_data['BB_lower_10']
  stock_data['KAMA_10'] = ta.KAMA(stock_data['Close -10d'], timeperiod=5)
  stock_data['MA_10'] = ta.MA(stock_data['Close -10d'], timeperiod=5)
  stock_data['TRIMA_10'] = ta.TRIMA(stock_data['Close -10d'], timeperiod=5)
  stock_data['ADXR_10'] = ta.ADXR(stock_data['High -10d'], stock_data['Low -10d'], stock_data['Close -10d'], timeperiod=5)
  stock_data['MINUS_DI_10'] = ta.MINUS_DI(stock_data['High -10d'], stock_data['Low -10d'], stock_data['Close -10d'], timeperiod=5)
  stock_data['MINUS_DM_10'] = ta.MINUS_DM(stock_data['High -10d'], stock_data['Low -10d'], timeperiod=5)
  stock_data['DEMA_10'] = ta.DEMA(stock_data['Close -10d'], timeperiod=5)
  stock_data["EMA_10"] = ta.EMA(stock_data["Close -10d"], timeperiod=5)
  stock_data['HT_TRENDLINE_10'] = stock_data["HT_TRENDLINE"].pct_change(periods=5)
  stock_data['MIDPOINT_10'] = ta.MIDPOINT(stock_data['Close -10d'], timeperiod=5)
  stock_data['ADX_10'] = ta.ADX(stock_data["High -10d"], stock_data["Low -10d"], stock_data["Close -10d"], timeperiod=5)
  stock_data["RSI_10"] = ta.RSI(stock_data["Close -10d"], timeperiod=5)
  stock_data["MIDPOINT_30_10"] = stock_data["MIDPOINT_10"].pct_change(periods=5)
  stock_data["atr_10"] = ta.ATR(stock_data["High -10d"], stock_data["Low -10d"], stock_data["Close -10d"], timeperiod=5)
  stock_data["SMA_10"] = ta.SMA(stock_data["Close -10d"], timeperiod=5)
  stock_data["ROC_10"] = ta.ROC(stock_data["Close -10d"], timeperiod=5)
  stock_data["CCI_10"] = ta.CCI(stock_data["High -10d"], stock_data["Low -10d"], stock_data["Close -10d"], timeperiod=5)
  stock_data["NATR_10"] = ta.NATR(stock_data["High -10d"], stock_data["Low -10d"], stock_data["Close -10d"], timeperiod=5)
  stock_data["TRIX_10"] = ta.TRIX(stock_data["Close -10d"], timeperiod=5)

  #30 day
  stock_data['BB_upper_30'], stock_data['BB_middle_30'], stock_data['BB_lower_30'] = ta.BBANDS(stock_data['Close -30d'], timeperiod=5, nbdevup=0, nbdevdn=2, matype=0)
  stock_data["bband_width_30"] = stock_data['BB_upper_30'] - stock_data['BB_lower_30']
  stock_data['KAMA_30'] = ta.KAMA(stock_data['Close -30d'], timeperiod=5)
  stock_data['MA_30'] = ta.MA(stock_data['Close -30d'], timeperiod=5)
  stock_data['TRIMA_30'] = ta.TRIMA(stock_data['Close -30d'], timeperiod=5)
  stock_data['ADXR_30'] = ta.ADXR(stock_data['High -30d'], stock_data['Low -30d'], stock_data['Close -30d'], timeperiod=5)
  stock_data['MINUS_DI_30'] = ta.MINUS_DI(stock_data['High -30d'], stock_data['Low -30d'], stock_data['Close -30d'], timeperiod=5)
  stock_data['MINUS_DM_30'] = ta.MINUS_DM(stock_data['High -30d'], stock_data['Low -30d'], timeperiod=5)
  stock_data['DEMA_30'] = ta.DEMA(stock_data['Close -30d'], timeperiod=5)
  stock_data["EMA_30"] = ta.EMA(stock_data["Close -30d"], timeperiod=5)
  stock_data['HT_TRENDLINE_30'] = stock_data["HT_TRENDLINE"].pct_change(periods=5)
  stock_data['MIDPOINT_30'] = ta.MIDPOINT(stock_data['Close -30d'], timeperiod=5)
  stock_data['ADX_30'] = ta.ADX(stock_data["High -30d"], stock_data["Low -30d"], stock_data["Close -30d"], timeperiod=5)
  stock_data["RSI_30"] = ta.RSI(stock_data["Close -30d"], timeperiod=5)
  stock_data["MIDPOINT_30_30"] = stock_data["MIDPOINT_30"].pct_change(periods=5)
  stock_data["atr_30"] = ta.ATR(stock_data["High -30d"], stock_data["Low -30d"], stock_data["Close -30d"], timeperiod=5)
  stock_data["SMA_30"] = ta.SMA(stock_data["Close -30d"], timeperiod=5)
  stock_data["ROC_30"] = ta.ROC(stock_data["Close -30d"], timeperiod=5)
  stock_data["CCI_30"] = ta.CCI(stock_data["High -30d"], stock_data["Low -30d"], stock_data["Close -30d"], timeperiod=5)
  stock_data["NATR_30"] = ta.NATR(stock_data["High -30d"], stock_data["Low -30d"], stock_data["Close -30d"], timeperiod=5)
  stock_data["TRIX_30"] = ta.TRIX(stock_data["Close -30d"], timeperiod=5)

  #Creating the label
  x = yf.Ticker(stock).history(start = change_months(startdate, 1), end = change_months(enddate, 1))["Close"]
  l = x.tolist()

  if stock_data.shape[0] > len(l):
    nan_count = stock_data.shape[0] - len(l)
    for i in range(nan_count):
      l.append(float('nan'))

  if stock_data.shape[0] < len(l):
    q = len(l)-stock_data.shape[0]
    for i in range(q):
      l.remove(l[-1])


  stock_data["future"] = l
  stock_data["future_pct_change"] = (stock_data["future"]-stock_data["Close"])/stock_data["Close"]
  stock_data["label"] = np.where(stock_data["future_pct_change"] > 0.05, 1,
                          np.where(stock_data["future_pct_change"] < -0.05, -1, 0)
                      )


  # Replace infinite values with NaN
  stock_data[np.isinf(stock_data)] = np.nan
  # Drop all Nan values
  stock_data = stock_data.dropna()


  stock_data = stock_data.drop(columns=["future","future_change"])#"one_mo_change"

  if shuffle == True:
    #shuffling the data
    stock_data = stock_data.sample(frac=1, random_state = 1)

  label = pd.DataFrame()
  label["label"] = stock_data.pop("label")

  return stock_data, label

#📌 pred_data function requires proof reading to ensure data created is accurate.
def pred_data(stock,end_,):
  start_ = change_days(end_, -150)
  pred = yf.Ticker(stock).history(start=start_, end=end_)


  for i in range(1, 60):
      pred[f"Close -{i}d"] = pred["Close"].shift(i)
      pred[f"High -{i}d"] = pred["High"].shift(i)
      pred[f"Low -{i}d"] = pred["Low"].shift(i)

  #Technical indicators
  pred['BB_upper'], pred['BB_middle'], pred['BB_lower'] = ta.BBANDS(pred['Close'], timeperiod=20, nbdevup=0, nbdevdn=2, matype=0)
  pred['KAMA'] = ta.KAMA(pred['Close'], timeperiod=30)
  pred['MA'] = ta.MA(pred['Close'], timeperiod=30)
  pred['TRIMA'] = ta.TRIMA(pred['Close'], timeperiod=30)
  pred['ADXR'] = ta.ADXR(pred['High'], pred['Low'], pred['Close'], timeperiod=14)
  pred['APO'] = ta.APO(pred['Close'], fastperiod=12, slowperiod=26)
  pred['MINUS_DI'] = ta.MINUS_DI(pred['High'], pred['Low'], pred['Close'], timeperiod=14)
  pred['MINUS_DM'] = ta.MINUS_DM(pred['High'], pred['Low'], timeperiod=14)
  pred['PPO'] = ta.PPO(pred['Close'], fastperiod=12, slowperiod=26, matype=0)
  pred['ADOSC'] = ta.ADOSC(pred['High'], pred['Low'], pred['Close'], pred['Volume'], fastperiod=3, slowperiod=10)
  pred['HT_DCPERIOD'] = ta.HT_DCPERIOD(pred['Close'])
  pred['TYPPRICE'] = ta.TYPPRICE(pred['High'], pred['Low'], pred['Close'])
  pred['WCLPRICE'] = ta.WCLPRICE(pred['High'], pred['Low'], pred['Close'])
  pred['DEMA'] = ta.DEMA(pred['Close'], timeperiod=30)##
  pred["EMA"] = ta.EMA(pred["Close"], timeperiod=14)  ##
  pred['HT_TRENDLINE'] = ta.HT_TRENDLINE(pred['Close'])##
  pred['MIDPOINT'] = ta.MIDPOINT(pred['Close'], timeperiod=14)##
  pred['ADX'] = ta.ADX(pred["High"], pred["Low"], pred["Close"], timeperiod=14)  ##
  pred["RSI"] = ta.RSI(pred["Close"], timeperiod=14)##
  pred["OBV"] = ta.OBV(pred["Close"], pred["Volume"])##
  pred["MIDPOINT_30"] = pred["MIDPOINT"].pct_change(periods=30)
  pred['SAR'] = ta.SAR(pred['High'], pred['Low'], acceleration=0.02, maximum=0.2)
  pred['macd'], pred['macd_signal'], pred['macd_diff'] = ta.MACD(pred['Close'], fastperiod=12, slowperiod=26, signalperiod=9)
  pred["bband_width"] = pred['BB_upper'] - pred['BB_lower']
  pred['stochastic oscillator %K'], pred['stochastic oscillator %D'] = ta.STOCH(pred['High'], pred['Low'], pred['Close'], fastk_period=5, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)
  pred["atr"] = ta.ATR(pred["High"], pred["Low"], pred["Close"], timeperiod=14)
  pred["SMA"] = ta.SMA(pred["Close"], timeperiod=14)
  pred["ROC"] = ta.ROC(pred["Close"], timeperiod=12)
  pred["CCI"] = ta.CCI(pred["High"], pred["Low"], pred["Close"], timeperiod=14)
  pred["AD"] = ta.AD(pred["High"], pred["Low"], pred["Close"], pred["Volume"])
  pred["NATR"] = ta.NATR(pred["High"], pred["Low"], pred["Close"], timeperiod=14)
  pred["TRIX"] = ta.TRIX(pred["Close"],timeperiod=30)

  #5 day
  pred['BB_upper_5'], pred['BB_middle_5'], pred['BB_lower_5'] = ta.BBANDS(pred['Close -5d'], timeperiod=5, nbdevup=0, nbdevdn=2, matype=0)
  pred["bband_width_5"] = pred['BB_upper_5'] - pred['BB_lower_5']
  pred['KAMA_5'] = ta.KAMA(pred['Close -5d'], timeperiod=5)
  pred['MA_5'] = ta.MA(pred['Close -5d'], timeperiod=5)
  pred['TRIMA_5'] = ta.TRIMA(pred['Close -5d'], timeperiod=5)
  pred['ADXR_5'] = ta.ADXR(pred['High -5d'], pred['Low -5d'], pred['Close -5d'], timeperiod=5)
  pred['MINUS_DI_5'] = ta.MINUS_DI(pred['High -5d'], pred['Low -5d'], pred['Close -5d'], timeperiod=5)
  pred['MINUS_DM_5'] = ta.MINUS_DM(pred['High -5d'], pred['Low -5d'], timeperiod=5)
  pred['DEMA_5'] = ta.DEMA(pred['Close -5d'], timeperiod=5)
  pred["EMA_5"] = ta.EMA(pred["Close -5d"], timeperiod=5)
  pred['HT_TRENDLINE_5'] = pred["HT_TRENDLINE"].pct_change(periods=5)
  pred['MIDPOINT_5'] = ta.MIDPOINT(pred['Close -5d'], timeperiod=5)
  pred['ADX_5'] = ta.ADX(pred["High -5d"], pred["Low -5d"], pred["Close -5d"], timeperiod=5)
  pred["RSI_5"] = ta.RSI(pred["Close -5d"], timeperiod=5)
  pred["MIDPOINT_30_5"] = pred["MIDPOINT_5"].pct_change(periods=5)
  pred["atr_5"] = ta.ATR(pred["High -5d"], pred["Low -5d"], pred["Close -5d"], timeperiod=5)
  pred["SMA_5"] = ta.SMA(pred["Close -5d"], timeperiod=5)
  pred["ROC_5"] = ta.ROC(pred["Close -5d"], timeperiod=5)
  pred["CCI_5"] = ta.CCI(pred["High -5d"], pred["Low -5d"], pred["Close -5d"], timeperiod=5)
  pred["NATR_5"] = ta.NATR(pred["High -5d"], pred["Low -5d"], pred["Close -5d"], timeperiod=5)
  pred["TRIX_5"] = ta.TRIX(pred["Close -5d"], timeperiod=5)

  #10 day
  pred['BB_upper_10'], pred['BB_middle_10'], pred['BB_lower_10'] = ta.BBANDS(pred['Close -10d'], timeperiod=5, nbdevup=0, nbdevdn=2, matype=0)
  pred["bband_width_10"] = pred['BB_upper_10'] - pred['BB_lower_10']
  pred['KAMA_10'] = ta.KAMA(pred['Close -10d'], timeperiod=5)
  pred['MA_10'] = ta.MA(pred['Close -10d'], timeperiod=5)
  pred['TRIMA_10'] = ta.TRIMA(pred['Close -10d'], timeperiod=5)
  pred['ADXR_10'] = ta.ADXR(pred['High -10d'], pred['Low -10d'], pred['Close -10d'], timeperiod=5)
  pred['MINUS_DI_10'] = ta.MINUS_DI(pred['High -10d'], pred['Low -10d'], pred['Close -10d'], timeperiod=5)
  pred['MINUS_DM_10'] = ta.MINUS_DM(pred['High -10d'], pred['Low -10d'], timeperiod=5)
  pred['DEMA_10'] = ta.DEMA(pred['Close -10d'], timeperiod=5)
  pred["EMA_10"] = ta.EMA(pred["Close -10d"], timeperiod=5)
  pred['HT_TRENDLINE_10'] = pred["HT_TRENDLINE"].pct_change(periods=5)
  pred['MIDPOINT_10'] = ta.MIDPOINT(pred['Close -10d'], timeperiod=5)
  pred['ADX_10'] = ta.ADX(pred["High -10d"], pred["Low -10d"], pred["Close -10d"], timeperiod=5)
  pred["RSI_10"] = ta.RSI(pred["Close -10d"], timeperiod=5)
  pred["MIDPOINT_30_10"] = pred["MIDPOINT_10"].pct_change(periods=5)
  pred["atr_10"] = ta.ATR(pred["High -10d"], pred["Low -10d"], pred["Close -10d"], timeperiod=5)
  pred["SMA_10"] = ta.SMA(pred["Close -10d"], timeperiod=5)
  pred["ROC_10"] = ta.ROC(pred["Close -10d"], timeperiod=5)
  pred["CCI_10"] = ta.CCI(pred["High -10d"], pred["Low -10d"], pred["Close -10d"], timeperiod=5)
  pred["NATR_10"] = ta.NATR(pred["High -10d"], pred["Low -10d"], pred["Close -10d"], timeperiod=5)
  pred["TRIX_10"] = ta.TRIX(pred["Close -10d"], timeperiod=5)

  #30 day
  pred['BB_upper_30'], pred['BB_middle_30'], pred['BB_lower_30'] = ta.BBANDS(pred['Close -30d'], timeperiod=5, nbdevup=0, nbdevdn=2, matype=0)
  pred["bband_width_30"] = pred['BB_upper_30'] - pred['BB_lower_30']
  pred['KAMA_30'] = ta.KAMA(pred['Close -30d'], timeperiod=5)
  pred['MA_30'] = ta.MA(pred['Close -30d'], timeperiod=5)
  pred['TRIMA_30'] = ta.TRIMA(pred['Close -30d'], timeperiod=5)
  pred['ADXR_30'] = ta.ADXR(pred['High -30d'], pred['Low -30d'], pred['Close -30d'], timeperiod=5)
  pred['MINUS_DI_30'] = ta.MINUS_DI(pred['High -30d'], pred['Low -30d'], pred['Close -30d'], timeperiod=5)
  pred['MINUS_DM_30'] = ta.MINUS_DM(pred['High -30d'], pred['Low -30d'], timeperiod=5)
  pred['DEMA_30'] = ta.DEMA(pred['Close -30d'], timeperiod=5)
  pred["EMA_30"] = ta.EMA(pred["Close -30d"], timeperiod=5)
  pred['HT_TRENDLINE_30'] = pred["HT_TRENDLINE"].pct_change(periods=5)
  pred['MIDPOINT_30'] = ta.MIDPOINT(pred['Close -30d'], timeperiod=5)
  pred['ADX_30'] = ta.ADX(pred["High -30d"], pred["Low -30d"], pred["Close -30d"], timeperiod=5)
  pred["RSI_30"] = ta.RSI(pred["Close -30d"], timeperiod=5)
  pred["MIDPOINT_30_30"] = pred["MIDPOINT_30"].pct_change(periods=5)
  pred["atr_30"] = ta.ATR(pred["High -30d"], pred["Low -30d"], pred["Close -30d"], timeperiod=5)
  pred["SMA_30"] = ta.SMA(pred["Close -30d"], timeperiod=5)
  pred["ROC_30"] = ta.ROC(pred["Close -30d"], timeperiod=5)
  pred["CCI_30"] = ta.CCI(pred["High -30d"], pred["Low -30d"], pred["Close -30d"], timeperiod=5)
  pred["NATR_30"] = ta.NATR(pred["High -30d"], pred["Low -30d"], pred["Close -30d"], timeperiod=5)
  pred["TRIX_30"] = ta.TRIX(pred["Close -30d"], timeperiod=5)

  pred=pred.dropna()
  return pred.tail(1)

#tie-breaking
def tie_break(l, epsilon=0.001):
    seen = {}
    for i, x in enumerate(l):
        if x in seen:
            count = seen[x]
            l[i] += epsilon * count
            seen[x] += 1
        else:
            seen[x] = 1
    return l

def backtest(stock, date, mo_or_wk):#function will return how much the stock price changed in the coming month/week

  if mo_or_wk == "mo":
    enddate = change_months(date, 1)

    initial_price = yf.Ticker(stock).history(start = change_days(date, -5), end = date).tail(1)
    new_price = yf.Ticker(stock).history(start = date, end = enddate).tail(1)
    print(f'''{stock}
      initial close: {initial_price["Close"]},
      new close: {new_price["Close"]}
      ''')

    pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])
    print(f"💎 {stock} CHANGE:", pct_change, "\n")
    return pct_change

  elif mo_or_wk == "wk":
    enddate = change_days(date, 7)
    #print(enddate)
    #print(startdate)

    initial_price = yf.Ticker(stock).history(start = change_days(date, -5), end = date).tail(1)
    new_price = yf.Ticker(stock).history(start = date, end = enddate).tail(1)
    #print(initial_price["Close"], new_price["Close"])

    pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])

    return pct_change

def backtest_exit_rules(stock, date):#function will return how much the stock price changed in the coming month with exit rules
  enddate = change_months(date, 1)
  print(f"[------=====⬜◼️⬜◼️⬜ Backtesting {stock} ⬜◼️⬜◼️⬜=====--------]")

  initial_price = yf.Ticker(stock).history(start = change_days(date, -5), end = date).tail(1)
  new_price = yf.Ticker(stock).history(start = date, end = enddate).tail(1)

  order = new_price["Close"].to_list()
  max = float(initial_price["Close"].iloc[0])*1.1
  min = float(initial_price["Close"].iloc[0])*0.9
  pct_change = (float(new_price["Close"].iloc[0])-float(initial_price["Close"].iloc[0]))/float(initial_price["Close"].iloc[0])
  print(f"[--------- Percentage Change if Held: {pct_change} ---------]")


  for i in order:
    if i >= max:
      print(f"[------- ⏹️⏹️ {stock} Price rose 10% due to exit orders ⏹️⏹️ -------]")
      return 0.1
    elif i < min:
      print(f"[------- ⏹️⏹️ {stock }Price rose 10% due to exit orders ⏹️⏹️ -------]")
      return -0.1

  print(f'''{stock}
    initial close: {initial_price["Close"]},
    new close: {new_price["Close"]}
    ''')

  print(f"Percentage Change if Held: {pct_change}")
  return pct_change

def backtest2(stock, date, repititions):
  df = pd.DataFrame()
  dictt = {'-3':0, "-2": 1, "-1":2, "1":3, "2" : 4, "3" : 5 }
  verdicts = []
  n3 = [] #negative 3
  n2 = []
  n1 = []
  p1 = []
  p2 = []
  p3 = [] #positive 3
  p = [1,2,3]
  n = [-1,-2,-3]
  dates = []
  results = []
  pct_changes = []

  for i in range(repititions):
    print((i/repititions)*100,' % Done')
    verdict, proba = backtest_technical_model(stock, date)
    verdicts.append(int(verdict))
    n3.append(proba[0][0])
    n2.append(proba[0][1])
    n1.append(proba[0][2])
    p1.append(proba[0][3])
    p2.append(proba[0][4])
    p3.append(proba[0][5])
    dates.append(date)
    date = change_days(date, 1)
    pct_change = backtest(stock, date, "mo")
    pct_changes.append(pct_change)
    if pct_change >0 and verdict in p:
      results.append(1)
    elif pct_change < 0 and verdict in n:
      results.append(1)
    else:
      results.append(0)


  df['date'] = dates
  df["verdicts"] = verdicts
  df["pct_chng"] = pct_changes
  df["results"] = results
  df["n3"] = n3
  df["n2"] = n2
  df["n1"] = n1
  df["p1"] = p1
  df["p2"] = p2
  df["p3"] = p3
  return df

def technical_model(stock, date):
  X,y = data(stock, change_months(date, -48), date)
  vote.fit(X,y.values.ravel())
  pred_df = pred_data(stock, date)
  #print(pred_df)
  pred = vote.predict(pred_df)
  proba = vote.predict_proba(pred_df)
  print(pred, proba)
  if len(proba[0]) == 6:
    print(f"""
    🔹{stock.upper()}🔹{pred}🔹{'🔷Bull🔷' if pred == 3 else '🔷Bear🔷' if pred == -3 else '◽?◽'} 🔹Bull Probability:{proba[0][5]}
          """)
  else:
    print("🔻 Error occured 🔻 Less than 6 labels 🔻")
    return  pred, 0

  try:
    return  pred,proba
  except Exception:
    return  pred, 0

def backtest_technical_model(stock, date):
  X,y = data(stock, change_months(date, -48), date,shuffle = False)

  X = X.iloc[:-22]
  y = y.iloc[:-22]

  X = pd.concat([X,y], axis=1)
  X = X.sample(frac=1, random_state=1)
  y = X.pop("label")

  vote.fit(X,y.values.ravel())
  pred_df = pred_data(stock, date)
  #print(pred_df)
  pred = vote.predict(pred_df)
  proba = vote.predict_proba(pred_df)
  print(pred, proba)
  if len(proba[0]) == 6:
    print(f"""
    🔹{stock.upper()}🔹{pred}🔹{'🔷Bull🔷' if pred == 3 else '🔷Bear🔷' if pred == -3 else '◽?◽'} 🔹Bull Probability:{proba[0][5]}
          """)
  else:
    print("🔻 Error occured 🔻 Less than 6 labels 🔻")
    return  pred, 0

  try:
    return  pred,proba
  except Exception:
    return  pred, 0

def simple_rsi(prices, timeperiod=14):
    # Calculate price changes (delta)
    delta = prices.diff()

    # Separate gains and losses
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    # Calculate RS using Simple Moving Average (SMA)
    avg_gain = gain.rolling(window=timeperiod).mean()
    avg_loss = loss.rolling(window=timeperiod).mean()

    rs = avg_gain / avg_loss

    # Apply your formula
    rsi = 100 - (100 / (1 + rs))
    return rsi

In [ ]:
# NOT VERY USEFUL
def DCS(proba): #Directional Confidence Score
  scores = []
  for i in range(len(proba)):
    x = proba[i][0]
    y = proba[i][round((len(proba[0])/2)-0.01)] #get middle class. Round rounds down, but this is find cuz first index is 0 anyways
    z = proba[i][-1]
    score = (z-x)*(1-y)
    scores.append(float(score))
  return scores

#Loading Algorithms

In [ ]:
# FOR MOMENTUM MODEL

#ADA NOT READY
ADA = AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=0, class_weight='balanced', min_samples_split=15, max_depth=100, max_leaf_nodes=18), random_state=0, learning_rate=0.01)
RF = RandomForestClassifier(random_state = 1,class_weight='balanced')
GB = GradientBoostingClassifier(random_state=0, learning_rate=0.05, )
ET = ExtraTreesClassifier(random_state = 1,class_weight='balanced')
BGRF = BaggingClassifier(estimator = RandomForestClassifier(random_state = 1, class_weight='balanced'), random_state =1)
LGBM = lgb.LGBMClassifier(verbose=-1, random_state=1,class_weight='balanced')
KNN = KNeighborsClassifier(n_neighbors = 11)
SVM = SVC(random_state=1, class_weight='balanced')

vote = VotingClassifier(estimators=[('RF', RF),('GB', GB), ('ET', ET)], voting='soft')

print("✅ Done loading the model ✅")
vote
#vote_score = cross_val_score(vote, X,y,cv=5)

#On SPGI: Cross validation score:
#[0.90106007 0.89399293 0.90106007 0.89399293 0.87588652]

✅ Done loading the model ✅


VotingClassifier(estimators=[('RF',
                              RandomForestClassifier(class_weight='balanced',
                                                     random_state=1)),
                             ('GB',
                              GradientBoostingClassifier(learning_rate=0.05,
                                                         random_state=0)),
                             ('ET',
                              ExtraTreesClassifier(class_weight='balanced',
                                                   random_state=1))],
                 voting='soft')

#ROC Classification

In [ ]:
# ROC DATA
def ROC_data(stock, startdate, enddate, shuffle=True, clean = True, traceback=0, classification_type = 'binary'):
  stock_data = yf.Ticker(stock).history(start = startdate, end=enddate)
  stock_data = stock_data.rename(columns={'Close': 'Close-0d', 'High': 'High-0d', 'Low': 'Low-0d', 'Volume': 'Volume-0d'})

  #Closing prices on previous days:
  for i in range(1, 60):
          stock_data[f"Close-{i}d"] = stock_data["Close-0d"].shift(i)
          stock_data[f"High-{i}d"] = stock_data["High-0d"].shift(i)
          stock_data[f"Low-{i}d"] = stock_data["Low-0d"].shift(i)
          stock_data[f"Volume-{i}d"] = stock_data["Volume-0d"].shift(i)

  # Momentum Indicators
  stock_data["ROC"] = ta.ROC(stock_data["Close-0d"], timeperiod=5)
  stock_data[f"CCI"] = ta.CCI(stock_data[f"High-0d"], stock_data[f"Low-0d"], stock_data[f"Close-0d"], timeperiod=5)
  stock_data['OBV'] = ta.OBV(stock_data['Close-0d'], stock_data['Volume-0d'])
  stock_data['OB-ROC'] = ta.OBV(stock_data['Close-0d'], stock_data['ROC']) # <----- [ =========================== EXPERIMENTAL =========================== ]
  stock_data['stoch-ROC %K'], stock_data['stoch-ROC %D'] = ta.STOCHRSI(stock_data['Close-0d'], timeperiod = 14, fastk_period=2, fastd_period=3, fastd_matype=0) # <----- [ =========================== EXPERIMENTAL =========================== ]
  stock_data['ROC-PPO'] = ta.PPO(stock_data['ROC'], fastperiod=3, slowperiod=5)

  stock_data[f'stochastic oscillator %K'], stock_data[f'stochastic oscillator %D'] = ta.STOCH(
      stock_data[f'High-0d'], stock_data[f'Low-0d'], stock_data[f'Close-0d'],
      fastk_period=2, slowk_period=5, slowk_matype=0, slowd_period=2, slowd_matype=0
  )

  time_periods = [2,3,4,5,6,7]

  for tp in time_periods:
      stock_data[f'OBV_slope(tp={tp})'] = ta.LINEARREG_SLOPE(stock_data['OBV'], timeperiod=tp)
      stock_data[f'ROC_slope(tp={tp})'] = ta.LINEARREG_SLOPE(stock_data['ROC'], timeperiod=tp)
      stock_data[f'CCI_slope(tp={tp})'] = ta.LINEARREG_SLOPE(stock_data['CCI'], timeperiod=tp)
      stock_data[f'stoch_%K_slope(tp={tp})'] = ta.LINEARREG_SLOPE(stock_data['stochastic oscillator %K'], timeperiod=tp)
      stock_data[f'Close_0d_slope(tp={tp})'] = ta.LINEARREG_SLOPE(stock_data['Close-0d'], timeperiod=tp)
      stock_data[f'OB-ROC_slope(tp={tp})'] = ta.LINEARREG_SLOPE(stock_data['OB-ROC'], timeperiod=tp) # <----- [ =========================== EXPERIMENTAL =========================== ]

  for tp in time_periods:
      stock_data[f'OBV_slope(tp={tp})s_slope'] = ta.LINEARREG_SLOPE(stock_data[f'OBV_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'ROC_slope(tp={tp})s_slope'] = ta.LINEARREG_SLOPE(stock_data[f'ROC_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'CCI_slope(tp={tp})s_slope'] = ta.LINEARREG_SLOPE(stock_data[f'CCI_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'stoch_%K_slope(tp={tp})s_slope'] = ta.LINEARREG_SLOPE(stock_data[f'stoch_%K_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'Close_0d_slope(tp={tp})s_slope'] = ta.LINEARREG_SLOPE(stock_data[f'Close_0d_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'OB-ROC_slope(tp={tp})s_slope'] = ta.LINEARREG_SLOPE(stock_data[f'OB-ROC_slope(tp={tp})'], timeperiod=tp)# <----- [ =========================== EXPERIMENTAL =========================== ]

      stock_data[f'OBV_slope(tp={tp})s_ROC'] = ta.ROC(stock_data[f'OBV_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'ROC_slope(tp={tp})s_ROC'] = ta.ROC(stock_data[f'ROC_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'CCI_slope(tp={tp})s_ROC'] = ta.ROC(stock_data[f'CCI_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'stoch_%K_slope(tp={tp})s_ROC'] = ta.ROC(stock_data[f'stoch_%K_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'Close_0d_slope(tp={tp})s_ROC'] = ta.ROC(stock_data[f'Close_0d_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'OB-ROC_slope(tp={tp})s_ROC'] = ta.ROC(stock_data[f'OB-ROC_slope(tp={tp})'], timeperiod=tp)# <----- [ =========================== EXPERIMENTAL =========================== ]


  stock_data["future"] = stock_data['ROC'].shift(traceback)
  stock_data["future_change"] = (stock_data["future"]-stock_data[f"ROC"])

  if classification_type.lower() == 'roc_change':
    stock_data["label"] = np.where(stock_data["future_change"] > 0, 1,0)
  elif classification_type.lower() == 'roc':
    stock_data["label"] = np.where(stock_data['future'] > 0, 1, 0)

  else:
    print('Please Enter A Suitable Classification Type.')
    return 'Please Enter A Suitable Classification Type.', 'Please Enter A Suitable Classification Type.'


  # Replace infinite values with NaN
  stock_data[np.isinf(stock_data)] = np.nan
  # Drop all Nan values
  stock_data = stock_data.dropna()

  stock_data = stock_data.drop(columns=['future', 'future_change'])

  #Drop uncorrelated features
  stock_data = stock_data.drop(columns=[ 'Stock Splits', 'Dividends'])
  #stock_data = stock_data.drop(columns = drpcol)

  for i in range(1, 60):
    stock_data = stock_data.drop(columns=[f"Close-{i}d", f"High-{i}d", f"Low-{i}d", f'Volume-{i}d'])

  if shuffle == True:
    #shuffling the data
    stock_data = stock_data.sample(frac=1, random_state = 1)

  label = pd.DataFrame()
  label["label"] = stock_data.pop("label")

  if clean == True:
    iso = IsolationForest(random_state=0)
    iso.fit(stock_data)
    outliers = iso.predict(stock_data)
    stock_data = stock_data[outliers == 1]
    label = label[outliers==1]

  return stock_data, label

In [ ]:
clfs = [RF, LGBM, GB, ADA, vote, vote, ET, KNN, vote]

In [ ]:
stocks = ['aapl', 'wmt', 'pltr', 'ko']
performances = {}

for clf in clfs:

  accuracies = []

  for stock in stocks:
    date = "2026-01-01"
    pred = 'ROC'.upper()
    traceback = -5
    classification_type = 'roc_change'

    X,y = ROC_data(stock, change_months(date, -100), date, shuffle = False, traceback = traceback, classification_type= classification_type)

    print(X.shape, y.shape)

    X_test = X.iloc[-200:]
    y_test = y.iloc[-200:]

    X = X.iloc[:-205]
    y = y.iloc[:-205]

    X = pd.concat([X,y], axis=1)
    X_train = X.sample(frac=1, random_state=1)

    y_train = X_train.pop("label")

    print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

    clf.fit(X_train, y_train.values.ravel())
    y_pred = clf.predict(X_test)
    y_pred_proba = clf.predict_proba(X_test)

    print(f"{stock} Accuracy: ",clf.score(X_test, y_test))
    cm = confusion_matrix(y_test, y_pred)
    cf = classification_report(y_test, y_pred)
    print(cm,'\n',cf)

    accuracies.append(precision_score(y_test, y_pred, average='macro'))

  performances[str(clf)] = sum(accuracies)/len(accuracies)
  print(performances)

(2018, 122) (2018, 1)
(1813, 122) (1813,) (200, 122) (200, 1)
aapl Accuracy:  0.725
[[81 25]
 [30 64]] 
               precision    recall  f1-score   support

           0       0.73      0.76      0.75       106
           1       0.72      0.68      0.70        94

    accuracy                           0.72       200
   macro avg       0.72      0.72      0.72       200
weighted avg       0.72      0.72      0.72       200

(2006, 122) (2006, 1)
(1801, 122) (1801,) (200, 122) (200, 1)
wmt Accuracy:  0.74
[[73 25]
 [27 75]] 
               precision    recall  f1-score   support

           0       0.73      0.74      0.74        98
           1       0.75      0.74      0.74       102

    accuracy                           0.74       200
   macro avg       0.74      0.74      0.74       200
weighted avg       0.74      0.74      0.74       200

(1244, 122) (1244, 1)
(1039, 122) (1039,) (200, 122) (200, 1)
pltr Accuracy:  0.66
[[81 19]
 [49 51]] 
               precision    recall 

In [ ]:
df = pd.DataFrame()
df['Change'] = X_test['Close-0d'].shift(-5) - X_test['Close-0d']
df['ROC'] = X_test['ROC']
df['ROC_label'] = y_test
df['ROC_pred'] = y_pred
df['ROC_proba'] = [x[1] for x in y_pred_proba]
df['Price_label'] = np.where(df['Change'] > 0, 1, 0)
df['Price_pred'] = np.where((df['ROC_proba'] > 0.5) & (X_test['ROC']>-2), 1,
                            np.where(((1-df['ROC_proba']) > 0.5) & (X_test['ROC']<2), 0, 9))
df = df.dropna()
print(df.to_string())

                              Change        ROC  ROC_label  ROC_pred  ROC_proba  Price_label  Price_pred
Date                                                                                                    
2025-03-10 00:00:00-04:00   5.488861  -3.780474          1         1   0.820232            1           9
2025-03-11 00:00:00-04:00   3.318848   1.151787          1         0   0.366749            1           0
2025-03-12 00:00:00-04:00   2.916245   1.781734          1         0   0.231557            1           0
2025-03-13 00:00:00-04:00   5.076447  -0.571661          1         1   0.623362            1           1
2025-03-14 00:00:00-04:00  -0.049103   2.809306          0         0   0.268017            0           9
2025-03-17 00:00:00-04:00  -0.058929   3.786757          0         0   0.129969            0           9
2025-03-18 00:00:00-04:00   2.533310   2.290910          0         0   0.250326            1           9
2025-03-19 00:00:00-04:00  -0.520432   1.976827        

In [ ]:
p=[]
n=[]
for i in df['ROC']:
  if i > 0:
    p.append(i)
  else:
    n.append(i)
pdf = pd.DataFrame({'':p})
ndf = pd.DataFrame({'':n})
print(pdf.mean(), ndf.mean())

    3.137782
dtype: float64    -2.48315
dtype: float64


In [ ]:
print(confusion_matrix(df['Price_label'], df['Price_pred']), '\n', classification_report(df['Price_label'], df['Price_pred']))

[[28 20 52]
 [16 28 51]
 [ 0  0  0]] 
               precision    recall  f1-score   support

           0       0.64      0.28      0.39       100
           1       0.58      0.29      0.39        95
           9       0.00      0.00      0.00         0

    accuracy                           0.29       195
   macro avg       0.41      0.19      0.26       195
weighted avg       0.61      0.29      0.39       195



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
print(df.to_string())

                             Change         RSI  Label  y_pred  final_pred
Date                                                                      
2025-03-11 00:00:00-04:00 -1.107498   72.487237      0       0           9
2025-03-12 00:00:00-04:00 -0.143494   47.569488      0       1           9
2025-03-13 00:00:00-04:00  0.583092   35.106037      1       1           9
2025-03-14 00:00:00-04:00 -0.476212    3.702579      0       1           9
2025-03-17 00:00:00-04:00 -1.137047   35.732359      0       1           9
2025-03-18 00:00:00-04:00 -0.553940   31.966109      0       1           9
2025-03-19 00:00:00-04:00  0.719147   46.594185      1       1           9
2025-03-20 00:00:00-04:00  1.000984   63.157786      1       0           9
2025-03-21 00:00:00-04:00  1.652122   42.507382      1       0           0
2025-03-24 00:00:00-04:00  2.594788   27.412985      1       1           9
2025-03-25 00:00:00-04:00  2.973801   35.678386      1       1           9
2025-03-26 00:00:00-04:00

In [ ]:
df = pd.DataFrame({'RSI': list(X_test['RSI']), 'Pred':y_pred, 'Label': y_test.values.ravel()})
df['Correct'] = np.where(df['Pred']==df['Label'], True, False)
#df = df.drop(columns = ['Pred', 'Label'])
print(df.to_string())

rsi = list(df['RSI'])
correct = list(df['Correct'])

right = []
wrong = []

for i, x in enumerate(correct):
  if x:
    right.append(rsi[i])
  else:
    wrong.append(rsi[i])

right = pd.DataFrame({'Right':right})
wrong = pd.DataFrame({"Wrong":wrong})

print(f'''
[============== Right ==============]
  Median RSI: {float(right.median())}
  Mean RSI: {float(right.mean())}
  sd: {float(right.std())}

[============== Wrong ==============]
  Median RSI: {float(wrong.median())}
  Mean RSI: {float(wrong.mean())}
  sd: {float(wrong.std())}
''')

           RSI  Pred  Label  Correct
0    20.594489     1      1     True
1    39.326985     0      1    False
2    34.963778     0      1    False
3    44.605535     1      1     True
4    36.015236     1      1     True
5    28.688013     1      1     True
6    41.125659     1      1     True
7    41.608315     1      1     True
8    55.729034     0      0     True
9    74.270434     0      0     True
10   78.046042     0      0     True
11   59.919513     0      1    False
12   60.679134     0      0     True
13   50.060899     1      0    False
14   45.500186     1      0    False
15   56.092725     1      0    False
16   68.034137     0      0     True
17   49.732061     0      0     True
18   30.986188     1      1     True
19   28.070309     1      1     True
20   23.027851     1      1     True
21   61.300028     0      0     True
22   49.258245     0      0     True
23   49.203330     0      0     True
24   49.234780     0      0     True
25   50.906176     0      1    False
2

/tmp/ipykernel_2928/2585415485.py:23: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  Median: {float(right.median())}
/tmp/ipykernel_2928/2585415485.py:24: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  Mean: {float(right.mean())}
/tmp/ipykernel_2928/2585415485.py:25: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  sd: {float(right.std())}
/tmp/ipykernel_2928/2585415485.py:28: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  Median: {float(wrong.median())}
/tmp/ipykernel_2928/2585415485.py:29: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. U

In [ ]:
drpcol = list(X_train.columns)

print(len(drpcol))

importances = list(RF.feature_importances_)
sorted_importances = sorted(importances, reverse=True)

for i in range(100):
  drpcol.remove(f'{X.columns[importances.index(sorted_importances[i])]}')

print(len(drpcol))

122
22


In [ ]:
drpcol=[]

In [ ]:
col_names = list(X_train.columns)
importances = list(RF.feature_importances_)
sorted_importances = sorted(importances, reverse=True)
for i,x in enumerate(sorted_importances):
  index = importances.index(x)
  print(col_names[index], x)

NotFittedError: This RandomForestClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
stocks = ["AAPL", "TSLA", "JNJ", "JPM", "XOM", "NVDA", "WMT"]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

date = "2026-01-01"
pred = 'TRIX'.upper()
traceback = -1
classification_type = 'binary'

for stock in stocks:
  X,y = TRIX_data(stock, change_months(date, -100), date, shuffle = False, clean=False, traceback = traceback, classification_type= classification_type, )

  #print(X.shape, y.shape)

  X_test = X.loc['2022-09-01':'2025-11-01']
  y_test = y.loc['2022-09-01':'2025-11-01']

  X = X.iloc[:-850]
  y = y.iloc[:-850]

  X = pd.concat([X,y], axis=1)
  X_train = X.sample(frac=1, random_state=1)

  y_train = X_train.pop("label")

  #print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

  vote.fit(X_train, y_train.values.ravel())
  y_pred = vote.predict(X_test)
  y_pred_proba = vote.predict_proba(X_test)

  print("[-------------------------------- Accuracy: ",vote.score(X_test, y_test))
  cm = confusion_matrix(y_test, y_pred)
  cf = classification_report(y_test, y_pred)
  #print(cm,'\n',cf)

  ''' #THIS IS USED TO TURN PREDICT PROBAS INTO DATAFRAMES
  df = pd.DataFrame()
  d = {}

  for i in range(len(y_pred_proba[0])):
    d[f'L{i}'] = []

  for i in range(len(y_pred_proba)):
    for j in range(len(y_pred_proba[0])):
      d[f'L{j}'].append(y_pred_proba[i][j])

  for i in d:
      df[f'{i}'] = d[i]

  '''

  df = pd.DataFrame({"":y_pred})

  print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)
  print(f'{stock.upper()}_{pred.upper()}_{classification_type.upper()}({traceback}d).xlsx')
  df.to_excel(f'{stock.upper()}_{pred.upper()}_{classification_type.upper()}({traceback}d).xlsx', index=False)
  df.to_excel(f'/content/drive/MyDrive/SP/New TAx Intermediate Inputs/predicted class as features/{pred.upper()}/FEATURE=CLASS_{stock.upper()}_{pred.upper()}_{classification_type.upper()}({traceback}d).xlsx', index=False)

  #X_test[f'{pred.upper()}'][-800:].to_excel(f'{stock.upper()}_{pred.upper()}_IndicatorValue(today).xlsx', index=False)
  #print(f'{stock.upper()}_{pred.upper()}_IndicatorValue(today).xlsx')
  #X_test[f'{pred.upper()}'][-800:].to_excel(f'/content/drive/MyDrive/SP/New TAx Intermediate Inputs/{pred.upper()}/{stock.upper()}_{pred.upper()}_IndicatorValue(today).xlsx', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[-------------------------------- Accuracy:  0.8264150943396227
(1184, 201) (1184,) (795, 201) (795, 1)
AAPL_TRIX_BINARY(-1d).xlsx
[-------------------------------- Accuracy:  0.8226415094339623
(1184, 201) (1184,) (795, 201) (795, 1)
TSLA_TRIX_BINARY(-1d).xlsx
[-------------------------------- Accuracy:  0.7710691823899372
(1184, 201) (1184,) (795, 201) (795, 1)
JNJ_TRIX_BINARY(-1d).xlsx
[-------------------------------- Accuracy:  0.7987421383647799
(1184, 201) (1184,) (795, 201) (795, 1)
JPM_TRIX_BINARY(-1d).xlsx
[-------------------------------- Accuracy:  0.8062893081761007
(1184, 201) (1184,) (795, 201) (795, 1)
XOM_TRIX_BINARY(-1d).xlsx
[-------------------------------- Accuracy:  0.7823899371069183
(1184, 201) (1184,) (795, 201) (795, 1)
NVDA_TRIX_BINARY(-1d).xlsx
[-------------------------------- Accuracy:  0.7987421383647799
(1184, 201) (1184,) (795

In [ ]:
for stock in stocks:
  X_test[f'{pred.upper()}'][-800:].to_excel(f'{stock.upper()}_{pred.upper()}_IndicatorValue(today).xlsx', index=False)
  print(f'{stock.upper()}_{pred.upper()}_IndicatorValue(today).xlsx')


AAPL_RSI_IndicatorValue(today).xlsx
TSLA_RSI_IndicatorValue(today).xlsx
JNJ_RSI_IndicatorValue(today).xlsx
JPM_RSI_IndicatorValue(today).xlsx
XOM_RSI_IndicatorValue(today).xlsx
NVDA_RSI_IndicatorValue(today).xlsx
WMT_RSI_IndicatorValue(today).xlsx


In [ ]:
y_test

,label
Date,
2022-09-01 00:00:00-04:00,1
2022-09-02 00:00:00-04:00,1
2022-09-06 00:00:00-04:00,1
2022-09-07 00:00:00-04:00,1
2022-09-08 00:00:00-04:00,0
...,...
2025-10-27 00:00:00-04:00,0
2025-10-28 00:00:00-04:00,0
2025-10-29 00:00:00-04:00,1


In [ ]:
y_train.loc['2022-08']

,label
Date,
2022-08-03 00:00:00-04:00,0
2022-08-02 00:00:00-04:00,0
2022-08-10 00:00:00-04:00,1
2022-08-01 00:00:00-04:00,1
2022-08-05 00:00:00-04:00,0
2022-08-09 00:00:00-04:00,1
2022-08-08 00:00:00-04:00,1
2022-08-04 00:00:00-04:00,0


#AutoML

In [ ]:
!pip install tpot[sklearnex]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.1/215.1 kB 12.6 MB/s eta 0:00:00
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires dill<0.3.9,>=0.3.0, but you have dill 0.4.1 which is incompatible.


In [ ]:
import tpot
from tpot import TPOTClassifier

In [ ]:
stock = "tsla"
date = "2026-01-01"
X,y = momentum_data(stock, change_months(date, -60), date,shuffle = False)

X_test = X.iloc[-200:]
y_test = y.iloc[-200:]

X = X.iloc[:-222]
y = y.iloc[:-222]

X = pd.concat([X,y], axis=1)
X_train = X.sample(frac=1, random_state=1)

y_train = X_train.pop("label")

"Train: ", X_train.shape, y_train.shape, "Test:", X_test.shape, y_train.shape

('Train: ', (806, 87), (806,), 'Test:', (200, 87), (806,))

In [ ]:
TPOT = tpot.TPOTClassifier(
    scorers = 'accuracy',
    generations = 50, #how many interations per algorithm is tested
    population_size = 10,
    verbose=4,
    cv = 5,
    max_time_mins = 240,
    preprocessing=False,
    early_stop=5,
    )
TPOT.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/tpot/tpot_estimator/estimator.py:458: UserWarning: Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.
  warnings.warn("Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.")
/usr/local/lib/python3.12/dist-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39909 instead
  warnings.warn(
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:45331
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:39909/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43585'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:33083 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:33083
INFO:dis

Generation:  1
Best accuracy_score score: 0.4131124913733609


Generation:   4%|▍         | 2/50 [14:01<5:54:06, 442.63s/it]

Generation:  2
Best accuracy_score score: 0.44424507323058043


Generation:   6%|▌         | 3/50 [21:29<5:48:44, 445.20s/it]

...

Generation:  30
Best accuracy_score score: 0.5012575722720649


INFO:distributed.core:Event loop was unresponsive in Scheduler for 3.25s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts and instability.
INFO:distributed.core:Event loop was unresponsive in Scheduler for 3.25s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts and instability.
INFO:distributed.core:Event loop was unresponsive in Scheduler for 3.55s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts and instability.
INFO:distributed.core:Event loop was unresponsive in Scheduler for 3.55s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts and instability.
INFO:distributed.core:Event loop was unresponsive in Nanny for 3.63s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can c

Generation:  31
Best accuracy_score score: 0.5012575722720649
Early stop



INFO:distributed.scheduler:Retire worker addresses (stimulus_id='retire-workers-1771756875.136514') (0,)
INFO:distributed.nanny:Closing Nanny at 'tcp://127.0.0.1:43585'. Reason: nanny-close
INFO:distributed.nanny:Nanny asking worker to close. Reason: nanny-close
INFO:distributed.core:Received 'close-stream' from tcp://127.0.0.1:41486; closing.
INFO:distributed.scheduler:Remove worker addr: tcp://127.0.0.1:33083 name: 0 (stimulus_id='handle-worker-cleanup-1771756875.1535032')
ERROR:distributed.scheduler:Removing worker 'tcp://127.0.0.1:33083' caused the cluster to lose scattered data, which can't be recovered: {'ndarray-defbd73be9d08fd262eb937088bc047d', 'DataFrame-9a2020d7e71bfa867cc122e0fea7a809'} (stimulus_id='handle-worker-cleanup-1771756875.1535032')
INFO:distributed.scheduler:Lost all workers
INFO:distributed.nanny:Nanny at 'tcp://127.0.0.1:43585' closed.
INFO:distributed.scheduler:Closing scheduler. Reason: unknown
INFO:distributed.scheduler:Scheduler closing all comms


TPOTClassifier(cv=5, early_stop=5, max_time_mins=240, scorers='accuracy',
               search_space=<tpot.search_spaces.pipelines.sequential.SequentialPipeline object at 0x78c92cf79e50>,
               verbose=4)

In [ ]:
score = TPOT.fitted_pipeline_.score(X_test, y_test)
print(f"Test Score: {score}")

Test Score: 0.08


In [ ]:
# Print the entire pipeline structure
print(TPOT.fitted_pipeline_)

# To specifically see the hyperparameters of the final model:
best_steps = list(TPOT.fitted_pipeline_.named_steps.keys())
print(f"Model: {best_steps}")
try:
  for i in range(len(best_steps)):
    print(f"Hyperparameters: {TPOT.fitted_pipeline_.named_steps[best_steps[i]].get_params()}")
except:
  print(1)

Pipeline(steps=[('minmaxscaler', MinMaxScaler()),
                ('rfe',
                 RFE(estimator=ExtraTreesClassifier(bootstrap=np.True_,
                                                    criterion=np.str_('entropy'),
                                                    max_features=0.2736539257821,
                                                    min_samples_leaf=4,
                                                    min_samples_split=16,
                                                    n_jobs=1),
                     step=0.8740781381637)),
                ('featureunion-1',
                 FeatureUnion(transformer_list=[('skiptransformer',
                                                 SkipTransformer()),
                                                ('passthrough',
                                                 Passthro...
                               feature_types=None, feature_weights=None,
                               gamma=0.0048414610878, grow_policy=

In [ ]:
list(TPOT.fitted_pipeline_.named_steps.keys())

In [ ]:
Opted = RandomForestClassifier(
        bootstrap= np.True_,
        ccp_alpha= 0.0,
        class_weight= 'balanced',
        criterion= np.str_('entropy'),
        max_depth= None,
        max_features= 0.1317695107895,
        max_leaf_nodes= None,
        max_samples= None,
        min_impurity_decrease= 0.0,
        min_samples_leaf= 1,
        min_samples_split= 5,
        min_weight_fraction_leaf= 0.0,
        monotonic_cst= None,
        n_estimators= 128,
        n_jobs= 1,
        oob_score= False,
        random_state= None,
        verbose= 0,
        warm_start= False
)

In [ ]:
Opted.fit(X_train, y_train)
Opted.score(X_test, y_test)

0.26

In [ ]:
y_pred = Opted.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
cf = classification_report(y_test, y_pred)
print(cm, "\n",cf)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


ValueError: X has 95 features, but RandomForestClassifier is expecting 7896 features as input.